In [ ]:
#import sys
#! {sys.executable} -m pip install jupyter-black
import jupyter_black

jupyter_black.load()

In [ ]:
%%html
<!-- Define CSS styles for the notebook. -->
<style>
    /* Turn off margin and padding for Jupyter cells
     * since the HTML export will be one continuous report document. */
    .jp-Cell {
        padding: 0 !important;
    }
    .jp-Cell-outputWrapper {
        margin: 0 !important;
    }

    /* Make the empty output cells not take up space in the exported HTML */
    .jp-mod-noOutputs {padding: 0;}

    /* Set up the floating dropdown menu. */
    .menu-container {
        position: fixed;
        top: 20px;
        left: 20px;
        z-index: 9999;
    }
    .dropdown-menu {
        margin: 2px;
        margin-top: -2px;
        background-color: #F0F0F0;
    }

    div#rendered_cells {
        padding-top: 70px;
    }

    /* Make sure tab titles are the length of their text. */
    .lm-TabBar-tab {flex-basis: auto !important;}
</style>

In [ ]:
# This cell defines metadata that appears in the notebook footer
METADATA = {
    "title": "ACCESS User Report",
    "version": 1,
    "description": "Monthly report on allocation usage and compute job performance for ACCESS users.",
    "history": [
        ["1", "2026-07-27", "Initial Version."],
    ],
}

In [ ]:
# Install dependencies.
# Commented out since they only need to run once. Uncomment as needed.
# import sys
# ! {sys.executable} -m pip install duckdb
# ! {sys.executable} -m pip install xdmod-data[report] anywidget
# ! {sys.executable} -m pip install --force-reinstall --no-deps git+https://github.com/aaronweeden/xdmod-data.git@whoami

In [ ]:
# Import modules and set styles.
import base64
from calendar import monthrange
from datetime import date, datetime, timedelta
from IPython.display import display, Markdown, HTML
import ipywidgets as widgets
import json
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import sys
import threading
import time
from xdmod_data.warehouse import DataWarehouse
import xdmod_data.themes
from xdmod_data.report import header, footer, set_styles
from zoneinfo import ZoneInfo

pio.renderers.default = "plotly_mimetype+notebook"
pio.templates.default = "timeseries"


def exception_handler(exception_type, exception, traceback):
    print(exception, file=sys.stderr)


# get_ipython()._showtraceback = exception_handler
display(set_styles())

In [ ]:
# Define report date constants.
YEAR = 2026
MONTH = 7
MONTH_YEAR = datetime(YEAR, MONTH, 1).strftime("%B %Y")
START_DATE = date(YEAR, MONTH, 1).strftime("%Y-%m-%d")
END_DATE = date(YEAR, MONTH, monthrange(YEAR, MONTH)[1]).strftime("%Y-%m-%d")
DATA_END_DATE = min(
    END_DATE, datetime.now(ZoneInfo("America/New_York")).date().isoformat()
)
YEAR_MONTH = f"{YEAR}-{MONTH:02d}"

In [ ]:
# Get data about the current user.
XDMOD_HOST = "https://xdmod-dev.ccr.xdmod.org"
dw = DataWarehouse(XDMOD_HOST)
USER_DATA = {
    "person_id": 10332,
    "first_name": "Aaron",
    "last_name": "Weeden",
}

In [ ]:
# Display header.
display(header())
display(
    Markdown(
        f"# ACCESS Metrics User Report for {USER_DATA['first_name']} {USER_DATA['last_name']} — {MONTH_YEAR}"
    )
)

In [ ]:
# Prepare to number sections, tables, and figures.
# SECTIONS variable below will be of the form:
# {
#     'Section 1 Title': {
#         'number': 1,
#         'subsections': {},
#     },
#     'Section 2 Title': {
#         'number': 2,
#         'subsections': {
#             'Section 2.1 Title': {
#                 'number': 1,
#                 'subsections': {
#                     'Section 2.1.1 Title': {
#                         'number': 1,
#                         'subsections': {},
#                     },
#                 },
#                 ...
#             },
#         },
#     },
#     ...
# }
SECTIONS = {}
# CURRENT_SECTION_TITLES below will contain the title of each current
# section/subsection/subsubsection, etc., e.g.,
# ['Section 1 Title', 'Section 1.1 Title', 'Section 1.1.1 Title']
CURRENT_SECTION_TITLES = []
CURRENT_SECTION_NUMBER_STR = ""
CURRENT_TABLE_NUMBER = 0
CURRENT_FIGURE_NUMBER = 0


def display_section_title(title, level):
    global SECTIONS
    global CURRENT_SECTION_TITLES
    global CURRENT_TABLE_NUMBER
    global CURRENT_FIGURE_NUMBER
    global CURRENT_SECTION_NUMBER_STR
    current_section_level = len(CURRENT_SECTION_TITLES)
    current_section = SECTIONS
    for i in range(0, current_section_level - 1):
        t = CURRENT_SECTION_TITLES[i]
        current_section = current_section[t]["subsections"]
    if level == current_section_level:
        add_section_at_same_level(title, level, current_section)
    elif level == current_section_level + 1:
        if level > 1:
            t = CURRENT_SECTION_TITLES[-1]
            current_section = current_section[t]["subsections"]
        CURRENT_SECTION_TITLES.append(title)
        current_section_level += 1
        current_section[title] = {
            "number": 1,
            "subsections": {},
        }
    elif level < current_section_level:
        while level < current_section_level:
            CURRENT_SECTION_TITLES.pop()
            current_section_level -= 1
        current_section = SECTIONS
        for i in range(0, current_section_level - 1):
            t = CURRENT_SECTION_TITLES[i]
            current_section = current_section[t]["subsections"]
        add_section_at_same_level(title, level, current_section)
    else:
        raise RuntimeError(
            f"Section title is at wrong level — jumped from level {current_section_level} to level {level}."
        )
    hashes = "#" + ("#" * level)
    current_section = SECTIONS
    section_numbers = []
    for i in range(0, current_section_level):
        t = CURRENT_SECTION_TITLES[i]
        current_section_number = current_section[t]["number"]
        section_numbers.append(current_section_number)
        current_section = current_section[t]["subsections"]
    CURRENT_SECTION_NUMBER_STR = ".".join([str(i) for i in section_numbers])
    display(Markdown(f"{hashes} {CURRENT_SECTION_NUMBER_STR}. {title}"))


def add_section_at_same_level(title, level, current_section):
    global CURRENT_SECTION_TITLES
    global CURRENT_TABLE_NUMBER
    global CURRENT_FIGURE_NUMBER
    if title not in current_section:
        t = CURRENT_SECTION_TITLES[-1]
        current_section_number = current_section[t]["number"]
        current_section[title] = {
            "number": current_section_number + 1,
            "subsections": {},
        }
        if level == 1:
            CURRENT_TABLE_NUMBER = 0
            CURRENT_FIGURE_NUMBER = 0
    CURRENT_SECTION_TITLES[-1] = title

In [ ]:
display_section_title("Introduction", level=1)
display(Markdown(f"""
This report summarizes usage and performance information for the jobs you ran on ACCESS-allocated resources
in {MONTH_YEAR}. It also provides information for other users on your ACCESS project(s), at your institution,
and in the same field of science as your project(s) so you can compare your usage and performance to theirs.

The metrics in this report use a unit of **ACCESS Credit Equivalents**, which is a normalized unit used to compare
usage across different types of resources with different compute capabilities. One ACCESS Credit Equivalent is defined
as one CPU Hour on SDSC Expanse (an AMD EPYC 7742 based compute resource).

The source of data for this report is [ACCESS XDMoD](https://xdmod.access-ci.org), which has data collected from ACCESS
Resource Providers. Please note the following caveats about the data in this report:

- ACCESS XDMoD is intended as a historical data source rather than a real-time data source, and there is a multi-day
  delay in data being available to view in ACCESS XDMoD.
- Data are occasionally missing or incorrect, and not every ACCESS-allocated resource provides all of the types of
  data that ACCESS XDMoD collects.
- Data are often backfilled and corrected by the ACCESS Metrics team, meaning the data in this report may change if
  loaded later, and data may be inconsistent with what is available in ACCESS XDMoD. ACCESS XDMoD should be treated
  as more up-to-date than this report except where otherwise noted.
- This report document will exist for a limited time (approximately three months), but the data will persist in
  ACCESS XDMoD.
"""))

In [ ]:
section_data = []


def display_section(title, level, render_function):
    display_section_title(title, level=level)
    container_widget = widgets.VBox([get_loading_gif_html_widget()])
    display(container_widget)
    section_data.append(
        {
            "container_widget": container_widget,
            "render_function": render_your_data_section,
        }
    )

In [ ]:
def update_plot_layout(
    plot,
    title,
    xaxis_title,
    is_timeseries,
    group_by_values,
    is_legend_visible,
    src,
):
    new_layout = {
        "annotations": [
            {
                "name": "title",
                "text": title,
                "xref": "paper",
                "yref": "paper",
                "xanchor": "center",
                "yanchor": "bottom",
                "x": 0.5,
                "y": 1.05,
                "font": {
                    "color": "#000000",
                    "family": "Lucida Grande, Lucida Sans Unicode, Arial, Helvetica, sans-serif",
                    "size": 19,
                },
                "showarrow": False,
                "captureevents": True,
            },
            {
                "name": "subtitle",
                "text": "User =  Weeden, Aaron Michael - SUNY Buffalo",
                "xref": "paper",
                "yref": "paper",
                "xanchor": "center",
                "yanchor": "bottom",
                "x": 0.5,
                "y": 1,
                "showarrow": False,
                "captureevents": True,
                "font": {"color": "#5078a0", "size": 15},
            },
            {
                "name": "credits",
                "text": f"{START_DATE} to {DATA_END_DATE}  Src: {src}. Powered by XDMoD/Plotly JS",
                "font": {
                    "color": "#909090",
                    "size": 9.6,
                    "family": "Lucida Grande, Lucida Sans Unicode, Arial, Helvetica, sans-serif",
                },
                "xref": "paper",
                "yref": "paper",
                "xanchor": "right",
                "yanchor": "bottom",
                "x": 1,
                "y": 0,
                "xshift": 25,
                "yshift": -140,
                "showarrow": False,
            },
        ],
        "legend": {
            "title": None,
            "itemwidth": 40,
            "itemsizing": "constant",
            "bgcolor": "#ffffff",
            "borderwidth": 0,
            "xref": "container",
            "yref": "container",
            "traceorder": "reversed",
            "font": {
                "family": "Lucida Grande, Lucida Sans Unicode, Arial, Helvetica, sans-serif",
                "color": "#274b6d",
                "size": 15,
            },
            "xanchor": "center",
            "yanchor": "bottom",
            "x": 0.5,
            "y": 0.02,
            "orientation": "h",
            "visible": is_legend_visible,
        },
        "hovermode": "x unified",
        "hoverdistance": 1,
        "hoverlabel": {
            "align": "left",
            "bgcolor": "rgba(255, 255, 255, 0.8)",
            "font": {
                "size": 12.8,
                "color": "#333333",
                "family": "Lucida Grande, Lucida Sans Unicode, Arial, Helvetica, sans-serif",
            },
            "namelength": -1,
            "bordercolor": "#1199ff",
        },
        "barmode": "stack",
        "images": [],
        "margin": {"t": 45, "r": 25, "b": 25, "l": 80},
        "yaxis": {
            "automargin": True,
            "autorangeoptions": {"minallowed": 0, "maxallowed": None},
            "layer": "below traces",
            "title": {
                "text": "<b>ACCESS Credit Equivalents</b>",
                "font": {
                    "color": "#1199ff",
                    "size": 15,
                    "family": "'Lucida Grande', 'Lucida Sans Unicode', Arial, Helvetica, sans-serif",
                },
            },
            "exponentformat": "SI",
            "tickfont": {"size": 14, "color": "#606060"},
            "ticksuffix": " ",
            "tickprefix": None,
            "tickmode": "auto",
            "nticks": 10,
            "type": "linear",
            "rangemode": "tozero",
            "range": [0, None],
            "separatethousands": True,
            "overlaying": None,
            "linewidth": 2.75,
            "linecolor": "#c0d0e0",
            "side": "left",
            "anchor": "x",
            "autoshift": True,
            "gridwidth": 1.375,
            "gridcolor": "#c0c0c0",
            "zeroline": False,
        },
        "xaxis": {
            "automargin": True,
            "layer": "below traces",
            "title": {
                "text": f"<b>{xaxis_title}</b>",
                "standoff": 5,
                "font": {
                    "color": "#000000",
                    "size": 15,
                    "family": "'Lucida Grande', 'Lucida Sans Unicode', Arial, Helvetica, sans-serif",
                },
            },
            "tickfont": {"size": 14, "color": "#606060"},
            "tickmode": "array",
            "spikedash": "solid",
            "spikethickness": 1,
            "spikecolor": "#C0C0C0",
            "linewidth": 2.75,
            "linecolor": "#c0d0e0",
            "showgrid": False,
            "gridcolor": "#c0c0c0",
            "zeroline": False,
        },
        "height": 500,
    }
    if is_timeseries:
        new_layout["xaxis"]["tickformat"] = "%Y-%m-%d"
        new_layout["xaxis"]["type"] = "date"
        new_layout["xaxis"]["rangemode"] = "tozero"
        new_layout["xaxis"]["hoverformat"] = "%Y-%m-%d"
        # Calculate tick labels (same algorithm as XDMoD PHP).
        value_count = (
            date.fromisoformat(DATA_END_DATE) - date.fromisoformat(START_DATE)
        ).days
        dtick = max(value_count // 12, 1)
        tickvals = []
        last_idx = value_count - 1
        include_both_labels = False
        i = 0
        while i < value_count:
            if not include_both_labels and ((value_count - i) <= dtick):
                if (value_count - i) <= round(dtick * 0.30):
                    i = last_idx
                else:
                    include_both_labels = True
            tickvals.append(date.fromisoformat(START_DATE) + timedelta(days=i))
            if i != last_idx and include_both_labels:
                i = last_idx - dtick
            i += dtick
        new_layout["xaxis"]["tickvals"] = tickvals
    if group_by_values is not None:
        new_layout["xaxis"]["ticktext"] = group_by_values
        new_layout["xaxis"]["type"] = "category"
        new_layout["xaxis"]["nticks"] = 20
        new_layout["xaxis"]["categoryarray"] = group_by_values
        new_layout["xaxis"]["categoryorder"] = "array"
        new_layout["xaxis"]["dtick"] = 0
        new_layout["xaxis"]["tickangle"] = -90
        new_layout["xaxis"]["ticklabelposition"] = "outside left"
        new_layout["xaxis"]["tickvals"] = group_by_values
    plot.update_layout(new_layout)
    plot.data = tuple(reversed(plot.data))
    plot.show()


def display_metric_explorer_link(
    title,
    realm,
    group_by_id,
    is_timeseries,
    legend_type,
    sort_type,
):
    metric_explorer_config = {
        "trend_line": False,
        "y_axis": {"original0": {"title": "ACCESS Credit Equivalents"}},
        "swap_xy": False,
        "share_y_axis": False,
        "hide_tooltip": False,
        "show_remainder": False,
        "timeseries": is_timeseries,
        "title": title,
        "legend_type": legend_type,
        "font_size": 3,
        "show_filters": True,
        "show_warnings": True,
        "data_series": {
            "data": [
                {
                    "color": "auto",
                    "combine_type": "stack",
                    "display_type": "column",
                    "enabled": True,
                    "filters": {"data": [], "total": 0},
                    "group_by": group_by_id,
                    "has_std_err": False,
                    "ignore_global": False,
                    "line_type": "Solid",
                    "line_width": 2,
                    "log_scale": False,
                    "long_legend": True,
                    "metric": "total_ace",
                    "realm": realm,
                    "shadow": False,
                    "sort_type": sort_type,
                    "std_err": False,
                    "trend_line": False,
                    "value_labels": False,
                    "visibility": None,
                    "x_axis": False,
                    "z_index": 0,
                }
            ],
            "total": 1,
        },
        "aggregation_unit": "Day",
        "global_filters": {
            "data": [
                {
                    "id": f"person={USER_DATA['person_id']}",
                    "value_id": USER_DATA["person_id"],
                    "value_name": f"{USER_DATA["last_name"]}, {USER_DATA["first_name"][0]}",
                    "dimension_id": "person",
                    "checked": True,
                }
            ],
            "total": 1,
        },
        "timeframe_label": "User Defined",
        "start_date": START_DATE,
        "end_date": END_DATE,
        "start": 0,
        "limit": 10,
    }
    display(HTML(f"""
            <p>
                <a
                    target="_blank"
                    href="{XDMOD_HOST}/#main_tab_panel:metric_explorer?config={base64.b64encode(json.dumps(metric_explorer_config).encode("utf-8")).decode("utf-8")}"
                ><i class="fa fa-external-link-alt"></i> Open in ACCESS XDMoD</a>
            </p>
            <hr>
        """))


display_section_title("Your Compute Usage", 1)
with dw:
    total_ace = dw.get_data(
        duration=[START_DATE, END_DATE],
        realm="Jobs",
        metric="total_ace",
        dimension="None",
        dataset_type="aggregate",
        aggregation_unit="Month",
        filters={"User": str(USER_DATA["person_id"])},
    )["ACCESS Credit Equivalents Charged: Total (SU)"]
display(
    Markdown(
        f"You used **{total_ace:,.1f}** ACCESS Credit Equivalents in {MONTH_YEAR}."
    )
)

In [ ]:
def display_aggregate_bar_plot_section(
    section_title, realm, group_by_id, group_by_label, src
):
    display_section_title(section_title, 2)
    with dw:
        df = dw.describe_dimensions(realm)
        default_group_by_label = df.loc[group_by_id]["label"]
        df = (
            dw.get_data(
                duration=[START_DATE, END_DATE],
                realm=realm,
                metric="total_ace",
                dimension=group_by_id,
                dataset_type="aggregate",
                aggregation_unit="Month",
                filters={"User": str(USER_DATA["person_id"])},
            )
            .to_frame()
            .reset_index()
            .rename(columns={default_group_by_label: group_by_label})
        )
    plot = px.bar(
        df, x=group_by_label, y="ACCESS Credit Equivalents Charged: Total (SU)"
    )
    plot.update_traces(
        hovertemplate="ACCESS Credit Equivalents Charged: Total (SU): <b>%{y:,.1f}</b><extra></extra>"
    )
    title = f"ACCESS Credit Equivalents Charged by {section_title}"
    update_plot_layout(
        plot,
        title,
        xaxis_title=group_by_label,
        is_timeseries=False,
        group_by_values=df[group_by_label],
        is_legend_visible=False,
        src=src,
    )
    display_metric_explorer_link(
        title,
        realm=realm,
        group_by_id=group_by_id,
        is_timeseries=False,
        legend_type="off",
        sort_type="label_asc",
    )

In [ ]:
display_aggregate_bar_plot_section(
    section_title="By Resource",
    realm="Jobs",
    group_by_id="resource",
    group_by_label="Resource",
    src="ACDB",
)

In [ ]:
display_section_title("By Resource By Day", 2)
with dw:
    df = dw.get_data(
        duration=[START_DATE, END_DATE],
        realm="Jobs",
        metric="total_ace",
        dimension="Resource",
        dataset_type="timeseries",
        aggregation_unit="Day",
        filters={"User": str(USER_DATA["person_id"])},
    )
df = (
    df.loc[:, (df != 0).any(axis=0)]
    .replace(0, pd.NA)
    .reindex(pd.date_range(start=START_DATE, end=DATA_END_DATE))
    .stack()
    .reset_index()
    .rename(columns={"level_0": "Time", 0: "Value"})
)
plot = px.bar(df, x="Time", y="Value", color="Resource")
plot.update_traces(hovertemplate="%{fullData.name}: <b>%{y:,.1f}</b><extra></extra>")
title = "ACCESS Credit Equivalents Charged by Resource by Day"
update_plot_layout(
    plot,
    title,
    xaxis_title="",
    is_timeseries=True,
    group_by_values=None,
    is_legend_visible=True,
    src="ACDB",
)
display_metric_explorer_link(
    title,
    realm="Jobs",
    group_by_id="resource",
    is_timeseries=True,
    legend_type="bottom_center",
    sort_type="value_desc",
)

In [ ]:
display_section_title("Your Compute Job Performance", 1)

In [ ]:
def display_utilization_histogram(type_, group_by_id):
    display_aggregate_bar_plot_section(
        section_title=f"{type_} Utilization",
        realm="SUPREMM",
        group_by_id=group_by_id,
        group_by_label=f"{type_} Utilization",
        src="SUPREMM",
    )


display_utilization_histogram(type_="CPU", group_by_id="cpuuser")
display_utilization_histogram(type_="Memory", group_by_id="max_mem")
display_utilization_histogram(type_="GPU", group_by_id="gpu_usage_bucketid")

In [ ]:
display(
    Markdown(
        f'''If you have any questions, comments, or concerns, you can
[submit a ticket](https://support.access-ci.org/help-ticket) — choose Open a Ticket,
log in with your ACCESS account, and for "ACCESS User Support Issue" choose "XDMoD Question."'''
    )
)
display(footer(METADATA))